## LLM Architecture Configuration
We configure our model parameters to align with a standard autoregressive language modeling structure:
* **Vocabulary Size (`vocab_size`):** `30,000`
* **Hidden Dimension Size (`hidden_size`):** `768`
* **Causal Decoder Mode (`is_decoder`):** Set to `True` to apply causal (left-to-right) attention masking. This allows us to keep the target dimensions `(8, 30k) -> (8, 768)` consistent without conflict from BERT's bidirectional pre-training defaults.

## Step 1: Tokenization
We begin by preparing our raw input text string through sequence tokenization.

* **Training Sequence:** `"The bank of the river flooded"`

In [11]:
import torch
from transformers import AutoTokenizer, BertConfig, BertLMHeadModel

# Define the training text sequence
sentence = "The bank of the river flooded"

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Configure BERT as a causal decoder to enforce left-to-right attention masking
config = BertConfig.from_pretrained("bert-base-uncased")
config.is_decoder = True

# Load the pre-trained BERT model with the causal language modeling head
model = BertLMHeadModel.from_pretrained("bert-base-uncased", config=config)

# Tokenize input sequence with fixed-length padding and truncation
inputs = tokenizer(sentence, max_length=8, padding="max_length", truncation=True, return_tensors="pt")
print("Tokenized Inputs:", inputs)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertLMHeadModel LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenized Inputs: {'input_ids': tensor([[  101,  1996,  2924,  1997,  1996,  2314, 10361,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


## Step 2: Causal Modeling & Inference
We pass the tokenized sequence through the model to obtain predictions (logits) and compute the corresponding language modeling loss.

In [12]:
# Forward pass to calculate causal LM loss and logits
# Note: Hugging Face models internally shift labels to align next-token predictions
outputs = model(**inputs, labels=inputs["input_ids"])
loss = outputs.loss
logits = outputs.logits  # Shape: [Batch, Sequence_Length, Vocab_Size]

## Step 3: Extracting Hidden States (Embeddings)
To feed representations into our custom attention layers, we extract the raw `768`-dimensional hidden state activations from the final layer of the transformer backplane, bypassing the vocabulary projection head.

In [13]:
# Forward pass returning hidden states of the transformer encoder layers
outputs_with_hidden = model(**inputs, output_hidden_states=True)

# Extract raw representation embeddings from the last decoder layer
# Shape: [Batch, Sequence_Length, Hidden_Size] (e.g., [1, 8, 768])
last_hidden_state = outputs_with_hidden.hidden_states[-1]
print(f"Extracted Hidden State Shape (Embeddings): {last_hidden_state.shape}")

# Assign embeddings to a clean descriptor for downstream layers
x_embeddings = last_hidden_state

Extracted Hidden State Shape (Embeddings): torch.Size([1, 8, 768])


## Step 4: Verification with Custom Multi-Head Self-Attention
Finally, we feed our extracted `768`-dimensional BERT embeddings into our custom implementation of a `MultiHeadAttention` layer, utilizing `12` attention heads (dimension per head: $768 / 12 = 64$).

In [7]:
# Multi-head Attention
# B: Batch Size (e.g., 1 sequence)
# T: Sequence Length / Time Steps (e.g., 8 tokens)
# D: Feature Dimension / Hidden Size / Vocab Size (e.g., 30522 logits)
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model:int, num_heads:int, dropout: float = 0.0):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads
    self.head_dim = self.d_k  # Add alias to prevent AttributeError in any version

    # (QKt/sq(model))* V = (d_model, d_model)
    self.qkv_proj = nn.Linear(d_model, 3 * d_model)
    self.out_proj = nn.Linear(d_model, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x:torch.Tensor, is_causal: bool = False)->torch.Tensor:
    B, T, D = x.shape
    qkv = self.qkv_proj(x)
    q,k,v = qkv.chunk(3, dim=-1)
    q,k,v = map(lambda t: t.view(B, T, self.num_heads, self.d_k).transpose(1,2), (q,k,v))
    attention_pattern = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)

    # Apply causal mask only if is_causal is True
    if is_causal:
      causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
      attention_pattern = attention_pattern.masked_fill(~causal_mask, float('-inf'))

    # 5. Softmax along key sequence dimension
    attn_weights = F.softmax(attention_pattern, dim=-1)
    attn_weights = self.dropout(attn_weights)
    # 6. Weighted sum over values: (B, H, T, T) @ (B, H, T, d_k) -> (B, H, T, d_k)
    context = torch.matmul(attn_weights, v)
    context = context.transpose(1, 2).contiguous().view(B, T, D)

    # 8. Final linear projection
    return self.out_proj(context)

In [14]:
# Instantiate Multi-Head Attention layer with BERT architecture parameters
# Model dimension (d_model) = 768, Number of heads = 12
mha_768 = MultiHeadAttention(d_model=768, num_heads=12)

# Process BERT embeddings using causal self-attention
attention_output = mha_768(x_embeddings, is_causal=True)

# Print and verify incoming/outgoing structural tensor dimensions
print(f"Input shape: {x_embeddings.shape}")
print(f"Attention Output shape: {attention_output.shape}")

Input shape: torch.Size([1, 8, 768])
Attention Output shape: torch.Size([1, 8, 768])


### Shape Verification & Comparison
Let's verify that the output of our custom Multi-Head Attention layer maintains the exact structural properties of our input embeddings, confirming that the token embeddings have been contextualized without shape distortion.

In [15]:
# Compare shapes directly using assertions
assert x_embeddings.shape == attention_output.shape, f"Mismatch! Input shape {x_embeddings.shape} != Output shape {attention_output.shape}"

print("=== Dimensional Comparison ===")
print(f"Input Embeddings Shape:  {list(x_embeddings.shape)}  -> [Batch, Seq_Len, Hidden_Dim]")
print(f"Attention Output Shape:  {list(attention_output.shape)}  -> [Batch, Seq_Len, Hidden_Dim]")
print("\n✅ Success: The custom Multi-Head Attention layer successfully preserves the spatial and feature dimensions.")

=== Dimensional Comparison ===
Input Embeddings Shape:  [1, 8, 768]  -> [Batch, Seq_Len, Hidden_Dim]
Attention Output Shape:  [1, 8, 768]  -> [Batch, Seq_Len, Hidden_Dim]

✅ Success: The custom Multi-Head Attention layer successfully preserves the spatial and feature dimensions.
